In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 03. Сравнение моделей\n",
    "## Детальный анализ производительности всех обученных моделей"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import sys\n",
    "from pathlib import Path\n",
    "\n",
    "sys.path.append(str(Path.cwd().parent))\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import json\n",
    "from sklearn.metrics import roc_curve, auc\n",
    "\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Загружаем метрики всех моделей\n",
    "metrics_dir = Path('../results/metrics')\n",
    "\n",
    "all_metrics = []\n",
    "for metric_file in metrics_dir.glob('*_metrics.json'):\n",
    "    with open(metric_file, 'r') as f:\n",
    "        metrics = json.load(f)\n",
    "    all_metrics.append(metrics)\n",
    "\n",
    "df_metrics = pd.DataFrame(all_metrics)\n",
    "df_metrics = df_metrics.sort_values('accuracy', ascending=False)\n",
    "df_metrics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Визуализация сравнения точности\n",
    "plt.figure(figsize=(14, 6))\n",
    "\n",
    "models = df_metrics['model'].tolist()\n",
    "accuracies = df_metrics['accuracy'].tolist()\n",
    "\n",
    "bars = plt.bar(range(len(models)), accuracies, color='skyblue', edgecolor='navy')\n",
    "\n",
    "# Добавляем значения\n",
    "for i, (bar, acc) in enumerate(zip(bars, accuracies)):\n",
    "    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,\n",
    "             f'{acc:.4f}', ha='center', va='bottom', fontsize=9)\n",
    "\n",
    "plt.xlabel('Model')\n",
    "plt.ylabel('Accuracy')\n",
    "plt.title('Model Comparison by Accuracy')\n",
    "plt.xticks(range(len(models)), models, rotation=45, ha='right')\n",
    "plt.ylim([0.5, 1])\n",
    "plt.grid(axis='y', alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Сравнение F1-score для обоих классов\n",
    "if 'f1_human' in df_metrics.columns and 'f1_robot' in df_metrics.columns:\n",
    "    plt.figure(figsize=(14, 6))\n",
    "    \n",
    "    x = np.arange(len(models))\n",
    "    width = 0.35\n",
    "    \n",
    "    plt.bar(x - width/2, df_metrics['f1_human'], width, \n",
    "            label='Human', color='green', alpha=0.7)\n",
    "    plt.bar(x + width/2, df_metrics['f1_robot'], width, \n",
    "            label='Robot', color='red', alpha=0.7)\n",
    "    \n",
    "    plt.xlabel('Model')\n",
    "    plt.ylabel('F1 Score')\n",
    "    plt.title('F1 Scores by Class')\n",
    "    plt.xticks(x, models, rotation=45, ha='right')\n",
    "    plt.legend()\n",
    "    plt.grid(axis='y', alpha=0.3)\n",
    "    plt.tight_layout()\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Тепловая карта метрик\n",
    "metrics_to_show = ['accuracy', 'precision_human', 'recall_human', 'f1_human',\n",
    "                   'precision_robot', 'recall_robot', 'f1_robot']\n",
    "\n",
    "available_metrics = [m for m in metrics_to_show if m in df_metrics.columns]\n",
    "\n",
    "plt.figure(figsize=(12, 8))\n",
    "sns.heatmap(df_metrics[available_metrics].set_index(df_metrics['model']),\n",
    "            annot=True, cmap='YlOrRd', fmt='.4f', cbar_kws={'label': 'Score'})\n",
    "plt.title('Model Performance Metrics Heatmap')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Радарная диаграмма для топ-3 моделей\n",
    "from math import pi\n",
    "\n",
    "top3 = df_metrics.head(3)\n",
    "\n",
    "categories = ['Accuracy', 'Precision (H)', 'Recall (H)', 'F1 (H)',\n",
    "              'Precision (R)', 'Recall (R)', 'F1 (R)']\n",
    "N = len(categories)\n",
    "\n",
    "angles = [n / float(N) * 2 * pi for n in range(N)]\n",
    "angles += angles[:1]\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))\n",
    "\n",
    "colors = ['blue', 'green', 'orange']\n",
    "\n",
    "for idx, (_, row) in enumerate(top3.iterrows()):\n",
    "    values = [\n",
    "        row['accuracy'],\n",
    "        row.get('precision_human', 0),\n",
    "        row.get('recall_human', 0),\n",
    "        row.get('f1_human', 0),\n",
    "        row.get('precision_robot', 0),\n",
    "        row.get('recall_robot', 0),\n",
    "        row.get('f1_robot', 0)\n",
    "    ]\n",
    "    values += values[:1]\n",
    "    \n",
    "    ax.plot(angles, values, 'o-', linewidth=2, color=colors[idx], label=row['model'])\n",
    "    ax.fill(angles, values, alpha=0.1, color=colors[idx])\n",
    "\n",
    "ax.set_xticks(angles[:-1])\n",
    "ax.set_xticklabels(categories)\n",
    "ax.set_ylim(0, 1)\n",
    "ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))\n",
    "ax.set_title('Radar Chart - Top 3 Models', size=14, pad=20)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# График скорость-точность (если есть данные о времени)\n",
    "if 'inference_time' in df_metrics.columns:\n",
    "    plt.figure(figsize=(12, 8))\n",
    "    \n",
    "    # Определяем цвета по типу модели\n",
    "    colors = []\n",
    "    for model in df_metrics['model']:\n",
    "        if any(x in model.lower() for x in ['forest', 'xgboost', 'catboost', 'logistic']):\n",
    "            colors.append('blue')\n",
    "        elif any(x in model.lower() for x in ['cnn', 'lstm', 'hybrid']):\n",
    "            colors.append('red')\n",
    "        else:\n",
    "            colors.append('green')\n",
    "    \n",
    "    plt.scatter(df_metrics['inference_time'] * 1000, df_metrics['accuracy'],\n",
    "               c=colors, s=200, alpha=0.6, edgecolors='black', linewidth=1)\n",
    "    \n",
    "    # Добавляем подписи\n",
    "    for i, row in df_metrics.iterrows():\n",
    "        plt.annotate(row['model'], \n",
    "                    (row['inference_time'] * 1000, row['accuracy']),\n",
    "                    xytext=(5, 5), textcoords='offset points', fontsize=8)\n",
    "    \n",
    "    plt.xlabel('Inference Time (ms)')\n",
    "    plt.ylabel('Accuracy')\n",
    "    plt.title('Speed-Accuracy Trade-off')\n",
    "    plt.grid(True, alpha=0.3)\n",
    "    \n",
    "    # Легенда\n",
    "    from matplotlib.patches import Patch\n",
    "    legend_elements = [\n",
    "        Patch(facecolor='blue', alpha=0.6, label='Traditional ML'),\n",
    "        Patch(facecolor='red', alpha=0.6, label='Deep Learning'),\n",
    "        Patch(facecolor='green', alpha=0.6, label='Ensemble')\n",
    "    ]\n",
    "    plt.legend(handles=legend_elements)\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Матрицы ошибок для топ-3 моделей\n",
    "fig, axes = plt.subplots(1, 3, figsize=(15, 4))\n",
    "\n",
    "for idx, (_, row) in enumerate(top3.iterrows()):\n",
    "    if 'confusion_matrix' in row and row['confusion_matrix']:\n",
    "        cm = np.array(row['confusion_matrix'])\n",
    "        \n",
    "        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',\n",
    "                   xticklabels=['Human', 'Robot'],\n",
    "                   yticklabels=['Human', 'Robot'],\n",
    "                   ax=axes[idx])\n",
    "        axes[idx].set_title(f\"{row['model']}\\nAcc: {row['accuracy']:.4f}\")\n",
    "        axes[idx].set_xlabel('Predicted')\n",
    "        axes[idx].set_ylabel('True')\n",
    "\n",
    "plt.suptitle('Confusion Matrices - Top 3 Models', fontsize=14)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Статистический тест для сравнения моделей (McNemar's test)\n",
    "from statsmodels.stats.contingency_tables import mcnemar\n",
    "\n",
    "# Здесь нужно загрузить предсказания моделей на тестовом наборе\n",
    "# и провести попарное сравнение\n",
    "\n",
    "print(\"Для проведения статистического теста МакНимара необходимо загрузить\")\n",
    "print(\"предсказания всех моделей на тестовом наборе данных.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Выводы и рекомендации\n",
    "print(\"=\" * 60)\n",
    "print(\"ВЫВОДЫ ПО СРАВНЕНИЮ МОДЕЛЕЙ\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "best_model = df_metrics.iloc[0]\n",
    "print(f\"\\n🏆 Лучшая модель по accuracy: {best_model['model']}\")\n",
    "print(f\"   Accuracy: {best_model['accuracy']:.4f}\")\n",
    "\n",
    "if 'f1_human' in best_model:\n",
    "    print(f\"   F1 (Human): {best_model['f1_human']:.4f}\")\n",
    "    print(f\"   F1 (Robot): {best_model['f1_robot']:.4f}\")\n",
    "\n",
    "print(\"\\n📊 Рейтинг моделей:\")\n",
    "for i, (_, row) in enumerate(df_metrics.iterrows()):\n",
    "    print(f\"   {i+1}. {row['model']}: {row['accuracy']:.4f}\")\n",
    "\n",
    "print(\"\\n💡 Рекомендации:\")\n",
    "print(\"   1. Для высокой точности используйте:\", df_metrics.iloc[0]['model'])\n",
    "if len(df_metrics) > 1:\n",
    "    print(\"   2. Для быстрого инференса используйте:\", \n",
    "          df_metrics[df_metrics['inference_time'] == df_metrics['inference_time'].min()]['model'].values[0]\n",
    "          if 'inference_time' in df_metrics.columns else \"(нет данных о скорости)\")\n",
    "print(\"   3. Рассмотрите ансамбль топ-3 моделей для повышения точности\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}